# Small Molecules

This notebook embeds real molecules from SMILES, annotates them with embpy molecule resources, and compares the resulting molecular embedding spaces.


In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display

from embpy import BioEmbedder, pl, tl

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

embedder = BioEmbedder(device="auto", organism="human")


def compact_obs(adata: ad.AnnData, prefixes: tuple[str, ...], base: list[str] | None = None) -> pd.DataFrame:
    base = base or []
    cols = [c for c in base if c in adata.obs.columns]
    cols += [c for c in adata.obs.columns if c.startswith(prefixes)]
    return adata.obs.loc[:, list(dict.fromkeys(cols))]


def short_embedding_key(key: str) -> str:
    parts = key.split("__")
    if len(parts) >= 3 and parts[0] == "X_emb":
        return f"X_{parts[2]}"
    return key


def feature_embeddings_as_obs(embedded: ad.AnnData, *, label_column: str = "label") -> ad.AnnData:
    """Convert real feature-aligned `.varm` embeddings to `.obsm` for plotting."""
    obs = embedded.var.copy()
    if label_column not in obs.columns:
        obs[label_column] = obs.index.astype(str)
    out = ad.AnnData(
        X=np.zeros((embedded.n_vars, 1), dtype=np.float32),
        obs=obs,
    )
    for key, matrix in embedded.varm.items():
        out.obsm[short_embedding_key(key)] = np.asarray(matrix, dtype=np.float32)
    out.uns["source_embeddings"] = {
        "varm_keys": list(embedded.varm.keys()),
        "note": "Matrices come directly from BioEmbedder.embed(..., output='anndata').",
    }
    return out

compounds = ["aspirin", "ibuprofen", "caffeine", "acetaminophen", "imatinib", "gefitinib"]
smiles = [
    "CC(=O)OC1=CC=CC=C1C(=O)O",
    "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
    "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "CC(=O)NC1=CC=C(O)C=C1",
    "CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5",
    "COC1=C(C=C2C(=C1)N=CN=C2NC3=CC(=C(C=C3)F)Cl)OCCCN4CCOCC4",
]
molecule_table = pd.DataFrame({"compound": compounds, "smiles": smiles})
display(molecule_table)


def rename_obsm_by_model(adata: ad.AnnData) -> ad.AnnData:
    for key in list(adata.obsm.keys()):
        adata.obsm[short_embedding_key(key)] = np.asarray(adata.obsm[key], dtype=np.float32)
    return adata


## Embed molecules

The Morgan fingerprint is computed locally with RDKit; ChemBERTa and MolFormer use the registered model wrappers.


In [ ]:
molecule_space = embedder.embed(
    molecule_table,
    entity_type="molecule",
    id_type="smiles",
    identifier_column="smiles",
    model=["morgan_fp", "chemberta2MTR", "molformer_base"],
    output="anndata",
    show_progress=True,
)
molecule_space = rename_obsm_by_model(molecule_space)
if "smiles" not in molecule_space.obs:
    molecule_space.obs["smiles"] = molecule_space.obs_names.astype(str)
if "compound" not in molecule_space.obs:
    molecule_space.obs["compound"] = compounds[: molecule_space.n_obs]

display(molecule_space)
print("obsm keys for plotting:", list(molecule_space.obsm.keys()))


## Annotate molecules


In [ ]:
molecule_space = tl.annotate_molecules(
    molecule_space,
    column="smiles",
    sources=["structural", "bioactivity", "ontology", "pathways", "xrefs"],
    copy=True,
)

display(compact_obs(molecule_space, ("mol_",), base=["compound", "smiles"]))
print("annotation stores:", [k for k in molecule_space.uns if "annotation" in k])


## Plot annotated molecule spaces


In [ ]:
color_key = "mol_logp" if "mol_logp" in molecule_space.obs else "compound"
pl.plot_embedding_space(
    molecule_space,
    obsm_key="X_morgan_fp",
    method="pca",
    color=color_key,
    annotate=True,
    annotate_col="compound",
    title="Morgan fingerprints colored by embpy molecule annotations",
)

if "mol_qed" in molecule_space.obs:
    pl.plot_embedding_space(
        molecule_space,
        obsm_key="X_chemberta2MTR",
        method="pca",
        color="mol_qed",
        annotate=True,
        annotate_col="compound",
        title="ChemBERTa embeddings colored by QED",
    )


## Compare molecule models


In [ ]:
k = min(2, molecule_space.n_obs - 1)
_, mean_overlap = tl.compute_knn_overlap(molecule_space, "X_morgan_fp", "X_chemberta2MTR", k=k)
print(f"Mean Morgan/ChemBERTa KNN overlap: {mean_overlap:.3f}")

pl.knn_overlap(molecule_space, obsm_keys=["X_morgan_fp", "X_chemberta2MTR", "X_molformer_base"], k=k)
pl.cross_embedding_correlation(molecule_space, "X_morgan_fp", "X_molformer_base")
pl.embedding_norms(molecule_space, obsm_keys=["X_morgan_fp", "X_chemberta2MTR", "X_molformer_base"])


## Save a reusable artifact


In [ ]:
molecule_space.write_h5ad(OUTPUT_DIR / "molecule_embeddings.h5ad")
print(OUTPUT_DIR / "molecule_embeddings.h5ad")
